# Gold Layer: Market Trends Fact Notebook
Analyzes macro-level price dynamics and listing volumes grouped by snapshot date, product category, and brand.

## 1. Setup and Imports
Load system modules and required database dependencies.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is available in system path
project_root = str(Path.cwd().parents[1])
if project_root not in sys.path:
    sys.path.append(project_root)

import polars as pl
from sqlalchemy import insert, select
from sqlalchemy.orm import Session

# Database configuration and model layer namespaces
from app.config import db_engine
from app.models import gold, silver

## 2. Query Silver Layer Listings
Retrieve daily listing price and category attributes from `silver.SilverCleanAd`.

In [ ]:
with db_engine.connect() as connection:
    df_silver_raw = pl.read_database(
        select(
            silver.SilverCleanAd.date,
            silver.SilverCleanAd.category,
            silver.SilverCleanAd.brand,
            silver.SilverCleanAd.price,
            silver.SilverCleanAd.ad_id
        ),
        connection=connection
    )

## 3. Aggregate Macro Market Trends
Group listings by date, category, and brand to calculate median price and total available supply volume.

In [ ]:
df_gold_market_trends = (
    df_silver_raw
    .group_by(['date', 'category', 'brand'])
    .agg(
        pl.col('price').median().alias('median_market_price'),
        pl.col('ad_id').count().alias('total_volume_available')
    )
)

## 4. Persist to Gold Market Trend Fact Table (`ft_gold_market_trends`)
Insert aggregate macro trends into `gold.FactMarketTrend`.

In [ ]:
if not df_gold_market_trends.is_empty():
    with Session(db_engine) as session:
        session.execute(
            insert(gold.FactMarketTrend), df_gold_market_trends.to_dicts()
        )
        session.commit()
        print(f"Successfully committed {len(df_gold_market_trends)} trend records.")
else:
    print("No trend data found to insert.")